# HUDM 5001 — Programming for Data Science
## Session 07 · Control Structures, Iterables, and Functions
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cgpan/hudm5001/blob/main/Week_07/07_Control_Structures_Iterables_Functions.ipynb)

[**Chenguang Pan**](https://cpan.ai/), [**Youmi Lab**](https://youmilab.ai/)
**Teachers College, Columbia University**


> ▶️ **How to open this notebook in Google Colab** — pick whichever is easiest:
> 1. **From Google Drive:** put this `.ipynb` in your Drive, right-click it → *Open with* → *Google Colaboratory*.
> 2. **Upload:** go to [colab.research.google.com](https://colab.research.google.com) → *File ▸ Upload notebook* → choose this file.
> 3. **From GitHub:** click the **Open in Colab** badge above (works once the notebook is pushed to the course repo).
>
> Then **run the Setup cell first** (▶ or `Shift+Enter`). Everything else runs top-to-bottom.  

About this notebook:  
This notebook is adapted from Prof. Youmi Suk's materials for HUDM 5001, Programming for Data Science ([original repository](https://github.com/youmilab/hudm5001)).  
Wording in this notebook was refined with Claude. I am responsible for its accuracy.  

Version: Oct.20.2026 10:00 am   

So far in this course we have stepped around two of the best-known words in programming, `if` and `for`, and we did that on purpose. NumPy and pandas are vectorized: you describe the result you want, and they do the repeating for you at machine speed. Some jobs still need you to spell out each step, though, like printing a custom report line for every group or running a simulation until it settles. Today you learn control structures, which let your code make decisions and repeat work, and iterables, the objects a loop walks through. Section 3 then gives you a first look at functions. You have been calling other people's functions all semester. This time you write one yourself with `def`, and the accumulator loop from earlier in the session becomes a tool you can reuse on any list.

---
## 1. Big Picture

### 🎯 Learning Objectives
By the end of this session you will be able to:
- Make decisions with `if` / `elif` / `else`, and read indentation as Python's way of showing which lines belong to which block.
- Pick between two values in one line with the conditional expression (`x if cond else y`).
- Repeat work with a `for` loop over any iterable: a list, a string, a `range`, or the `.items()` of a dictionary.
- Build the accumulator patterns (running totals, counters, best-so-far trackers) and loop with `enumerate` and `zip`.
- Use `while` to repeat until something changes, without getting stuck in an infinite loop.
- Steer a loop with `break` (stop the whole loop) and `continue` (skip to the next pass).
- Put loops to work on real data: a report line for each group, and a missing-value check on every column.
- Recognize the built-in functions you have been calling all along, and use the ones we held back until now (`sum`, `min`, `max`, `sorted`, `abs`). Look up any function with `help()`, and find more in modules such as `math`.
- Write your own function with `def` (parameters, a docstring, and a `return`), and explain why defining a function runs nothing until you call it.

### ✅ Prerequisites
- Sessions 01 to 06. The comparison and logical operators from Session 02 become the conditions inside every `if` and `while` today, and the containers from Session 01 are what we loop over.
- Run the Setup cell to import the libraries and set the data URLs.

### 🧩 Key Concepts
`if / elif / else`  `indentation`  `conditional expression`  `for`  `while`  `break`  `continue`  `accumulator`  `enumerate`  `zip`  `iterable`  `.items()`  `built-in function`  `help()`  `module`  `def`  `return`  `docstring`  `parameter / argument`

### 📖 Reading
- McKinney, *Python for Data Analysis*, §2.3 (Control Flow) and Chapter 3 (Built-in Data Structures)

### 🗺️ Today's Agenda
- 2.1 Why we need control flow
- 2.2 Decisions with `if` / `elif` / `else`
- 2.3 One-liners: the conditional expression
- 2.4 `for` loops over an iterable
- 2.5 Loop patterns: accumulators, `enumerate`, `zip`
- 2.6 `while` loops: repeat until told to stop
- 2.7 Steering a loop with `break` and `continue`
- 2.8 Loops meet data (iris and Titanic)
- 3.1 Functions you already use: the built-ins
- 3.2 Your first function with `def`

---
## ▶️ Setup — run this cell first
This imports the libraries, sets the web addresses of our two old friends — the **iris** flowers from Session 03 and the **Titanic** passengers from Session 04 — and grabs the `time` module for a small speed test near the end.

In [1]:
# === SETUP — run me first! ===
import numpy as np
import pandas as pd
import time     # we will time a loop against a vectorized line in section 2.10

# The classical course datasets, read straight from the public repo (no downloads).
IRIS_URL    = "https://raw.githubusercontent.com/cgpan/hudm5001/main/asset/iris.csv"
TITANIC_URL = "https://raw.githubusercontent.com/cgpan/hudm5001/main/asset/titanic.csv"

print("pandas", pd.__version__, "| NumPy", np.__version__)
print("Setup complete ✅")

pandas 2.2.3 | NumPy 2.1.3
Setup complete ✅


---
## 2. Content

### 2.1 Why control flow — the missing piece
Think about what you could *already* do by the end of Session 06: filter a table, summarize by group, join two datasets, query a database. All of it without a single `if` or `for`. That is because a boolean mask **is** a thousand tiny if-decisions, made for you at machine speed — `titanic["age"] > 60` quietly asks *"is this passenger over 60?"* 891 times.

What you could *not* do yet is **act** on a decision, or **repeat** your own steps. Session 02 taught you to *compute* a verdict as a boolean; `if` is what lets your program *do something* about it.

In [ ]:
score = 72

# Session 05 taught you to COMPUTE a decision...
print("passed?", score >= 60)

# ...but not to ACT on it. That is what `if` adds:
if score >= 60:
    print("Passed — nice work!")

Two ideas cover all of control flow:
- **Branching** — run this block *or* that one, depending on a condition (`if` / `elif` / `else`).
- **Repetition** — run a block many times (`for` over a collection, `while` a condition holds).

Everything else today (`break`, `continue`, iterators) is refinement of those two.

### 2.2 `if` / `elif` / `else` — decisions
The anatomy is three parts: the keyword `if`, a **condition** (any expression that evaluates to `True`/`False` — exactly the comparisons from Session 02), and a **colon**, followed by an **indented block**. The indentation is not decoration: it is Python's grammar for *"these lines belong to the `if`"*. Four spaces is the convention, and Colab indents for you when you type the colon and press Enter.

In [ ]:
score = 72
attendance = 0.9

# the Session-02 pass rule, now with consequences
if (score >= 60) and (attendance >= 0.8):
    print("Passed the course.")
else:
    print("Not this time — see the make-up policy.")

When there are more than two possibilities, chain conditions with **`elif`** ("else if"). Python checks the conditions **top to bottom** and runs the block of the *first* one that is `True` — then skips the rest of the ladder entirely. An optional final **`else`** is the catchall for "none of the above".

In [ ]:
score = 87

if score >= 90:
    grade = "A"
elif score >= 80:
    grade = "B"          # <- 87 lands here: the FIRST true condition wins
elif score >= 70:
    grade = "C"
else:
    grade = "below C"

print("score", score, "->", grade)

#### ✏️ Try It Yourself — Exercise 1
**Difficulty:** ★★☆☆☆  **Skills:** `if` / `elif` / `else`

Airlines label the Titanic's ports; we will label its fares. Given `fare = 31.0`, write a ladder that prints `"first-class territory"` when the fare is **at least 80**, `"comfortable"` when it is **at least 30**, `"steerage"` when it is **at least 10**, and `"bargain"` otherwise. Then change `fare` to `7.25` and run it again — which rung fires now?

In [3]:
fare = 7.25
# Your code here
if fare >=80:
  print('first-class territory')
elif fare >= 30:
  print('comfortable')
elif fare >= 10:
  print('steerage')
else:
  print('bargain')


bargain


<details>
<summary>▶ Show solution</summary>

```python
fare = 31.0

if fare >= 80:
    print("first-class territory")
elif fare >= 30:
    print("comfortable")       # <- 31.0 lands here
elif fare >= 10:
    print("steerage")
else:
    print("bargain")

# with fare = 7.25 the first three rungs are all False, so the else fires: "bargain"
```

</details>

### 2.3 One-liners — the conditional expression
Sometimes a branch is not about *doing* two different things, just about *picking* one of two **values**. Python has an expression form for that — it produces a value, so it can sit on the right side of `=`, inside an f-string, anywhere a value goes.

Syntax:  
```python
var_a = value_if_true if condition else value_if_false
```

In [6]:
score = 72
label = "pass" if score >= 60 else "fail"     # reads: label IS "pass" if ... otherwise "fail"
print(label)

# it slots straight into an f-string, too
# try this condition: print `senior` if one's age is at least 65; otherwise, print `adult``
age = 71
print(f"passenger status: {'try here'}.")

pass
passenger status: try here.


Both halves are required — `"pass" if score >= 60` alone is a syntax error, because an expression must produce a value in *every* case. Use the one-liner for simple picks; the moment you feel tempted to nest one inside another, switch back to a full `if`/`elif` ladder — future-you will thank you.

### 2.4 `for` loops — repetition over an iterable
A **`for` loop** walks through the items of a collection, one at a time: `for item in collection:` plus an indented block. Each pass, the **loop variable** (`item`) is refilled with the next element and the block runs again. Name it something singular and readable — `for program in programs` reads like English.

Syntax:  
```Python
for i in a_collection:              # <---- must end with a colon.
    your_operation_1_on_each_i      # <---- must have start with indented block
    your_operation_2                # <---- must have start with indented block if this line is still for each i

print("..")                         # a new line without indented block means the for loop ends and you start to run a new operation
```

In [7]:
programs = ["Applied Stats", "Learning Analytics", "Measurement"]

for program in programs:
    print(program, "->", len(program), "characters")

Applied Stats -> 13 characters
Learning Analytics -> 18 characters
Measurement -> 11 characters


Anything you can loop over is called an **iterable** — and you have been collecting them since Session 01. A string is an iterable of characters. And `range`, which looked slightly pointless back then, finally shows its true job: driving a loop without building a list first.

In [8]:
# a string is an iterable of characters
for ch in "iris":
    print(ch.upper())

I
R
I
S


In [9]:
# range drives a loop — this is what it was born to do
for i in range(1, 6):
    print(i, "squared is", i ** 2)

1 squared is 1
2 squared is 4
3 squared is 9
4 squared is 16
5 squared is 25


### 2.5 Loop patterns: accumulators, `enumerate`, `zip`
The single most useful loop shape is the **accumulator**: create a variable *before* the loop, update it *inside* the loop, use it *after*. This is precisely what `.sum()` and `.mean()` have been doing for you all along.

One new piece of shorthand: `total += x` means `total = total + x` — read it as "add `x` onto `total`".

In [10]:
scores = [88, 92, 79, 95, 84]

total = 1                 # 1. create BEFORE the loop
for s in scores:
    total += s            # 2. update INSIDE the loop, this line means: total = total + s
print("total:", total)    # 3. use AFTER the loop
print("mean :", total / len(scores))

total: 439
mean : 87.8


Swap the running total for a **counter** and add an `if`, and you have your first loop-plus-branch combination — the bread and butter of everyday programming.

In [ ]:
# how many scores are above 85?
count = 0
for s in scores:
    if s > 85:
        count += 1
print(count, "scores above 85")

Two companions you will use constantly:
- **`enumerate(xs)`** hands you the **position and the value** as a pair — no manual counter needed. Pass `start=1` to count like a human.
- **`zip(xs, ys)`** walks **two lists in step**, pairing them up element by element.

Both pairs unpack straight into two loop variables — the same `a, b = 3, 4` unpacking trick from Session 01.

In [11]:
names  = ["Ann", "Bob", "Cara", "Dan", "Eve"]
scores = [88, 92, 79, 95, 84]

# enumerate: position + value
for i, s in enumerate(scores, start=1):
    print(f"student {i}: {s}")

student 1: 88
student 2: 92
student 3: 79
student 4: 95
student 5: 84


A `zip()` function pairs up items from two or more iterables by position.

In [ ]:
print(list(zip(names, scores)))

# zip: two lists, walked in step
for name, s in zip(names, scores):
    print(f"{name} scored {s}")

One more accumulator variant rounds out the family: the **best-so-far tracker**. Instead of adding, you *compare* — keep a "champion" variable, and replace it whenever the loop meets something better. This is how `max()` works inside, and it shines when you need not just the best value but *who owns it*.

In [ ]:
# best-so-far: who has the top score?
best_name = ""
best_score = 0
scores = [88, 92, 79, 95, 84]
names  = ["Ann", "Bob", "Cara", "Dan", "Eve"]
for name, s in zip(names, scores):
    if s > best_score:        # a challenger beats the champion...
        best_score = s        # ...so both trackers update
        best_name = name
print(f"top score: {best_score}, by {best_name}")

#### ✏️ Try It Yourself — Exercise 2
**Difficulty:** ★★★☆☆  **Skills:** accumulator, loop + `if`, `enumerate`

Here are one week's step counts: `steps = [8200, 10500, 6400, 12000, 9300]`.
(a) Compute the **total** and the **mean** with an accumulator loop — no `sum()` allowed today.
(b) With a loop and an `if`, **count** the days above 9,000 steps.
(c) Use `enumerate(..., start=1)` to print one line per day, like `Day 1: 8200 steps`.

In [15]:
# Your code here
steps = [8200, 10500, 6400, 12000, 9300]
total=0
for i in steps:
  total=total+i
print(total,total/len(steps))
count=0
for i in steps:
  if i >9000:
    count=count+1
print(count)
for d,s in enumerate(steps,start=1):
  print(f'Day{d}:{s} steps')

46400 9280.0
3
Day1:8200 steps
Day2:10500 steps
Day3:6400 steps
Day4:12000 steps
Day5:9300 steps


<details>
<summary>▶ Show solution</summary>

```python
steps = [8200, 10500, 6400, 12000, 9300]

# (a) accumulator
total = 0
for s in steps:
    total += s
print("total:", total, "| mean:", total / len(steps))

# (b) counter + if
count = 0
for s in steps:
    if s > 9000:
        count += 1
print(count, "days above 9,000 steps")

# (c) enumerate
for day, s in enumerate(steps, start=1):
    print(f"Day {day}: {s} steps")
```

</details>

### 2.6 `while` loops — repeat until told to stop
A **`while` loop** re-runs its block *as long as* a condition stays `True`. Use it when you cannot know in advance **how many** repetitions you need — you only know when to stop. The condition is checked *before* each pass, so something inside the loop must eventually change it.

In [16]:
# keep doubling until we pass 10 — how many is that? we don't know in advance!
ix = 1
while ix < 10:
    ix = ix * 2
    print("ix is now", ix)
print("final:", ix)

ix is now 2
ix is now 4
ix is now 8
ix is now 16
final: 16


A more honest example: your savings grow 6% a year — **how many years until they double?** Nobody hands you the number of iterations; the condition discovers it.

In [17]:
balance = 1000
years = 0

while balance < 2000:
    balance = balance * 1.06     # one year of growth
    years += 1

print(f"doubled after {years} years, at ${round(balance, 2)}")

doubled after 12 years, at $2012.2


> ⚠️ **The infinite loop.** If the condition never becomes `False` — say you forget the `years += 1`-style update — the loop runs forever and the cell just spins. In Colab, click the cell's stop button (or *Runtime ▸ Interrupt execution*), find what never changed, and re-run. Every programmer has done this; the trick is recognizing the spin quickly.

**`for` or `while`?** If you are walking a collection you already have, use `for`. If you are repeating *until something happens*, use `while`. When either would work, `for` is usually the safer, clearer choice — it cannot loop forever.

#### ✏️ Try It Yourself — Exercise 3
**Difficulty:** ★★★☆☆  **Skills:** `while`, counter

A university's tuition is **$1,000** per credit and rises **5% each year**. Using a `while` loop, find how many years it takes for the price to **double** (reach $2,000 or more). Print the number of years and the final price, rounded to 2 decimals.

In [20]:
# Your code here
balance = 1000
years = 0
while balance < 2000:
  balance=balance*1.05
  years=years+1
print(f"doubles after {years} years, at ${round(balance, 2)}")

doubles after 15 years, at $2078.93


<details>
<summary>▶ Show solution</summary>

```python
price = 1000
years = 0

while price < 2000:
    price = price * 1.05
    years += 1

print(f"doubles after {years} years, at ${round(price, 2)}")
# 15 years — and the rule of 70 predicted 70 / 5 = 14. Close enough for a napkin.
```

</details>

### 2.7 `break` and `continue` — steering a loop
Two keywords redirect a running loop, and both almost always sit inside an `if`:
- **`break`** — abandon the **whole loop** right now. Perfect for searches: once you have found what you wanted, why keep looking?
- **`continue`** — abandon **this pass only** and jump to the next item. Perfect for skipping bad or irrelevant entries.

In [ ]:
# break: find the FIRST passenger fare over 100, then stop searching
fares = [7.25, 71.28, 8.05, 146.52, 11.13, 30.07]

for fare in fares:
    if fare > 100:
        print("found one:", fare)
        break                      # <- the loop ends here, items after are never seen
    print("checking", fare)

In [ ]:
# continue: skip the missing values, average the rest
readings = [7.0, None, 6.4, 5.8, None, 7.7]

total = 0
count = 0
for r in readings:
    if r is None:
        continue                   # <- skip this pass; the two lines below never run
    total += r
    count += 1

print(f"mean of {count} valid readings: {round(total / count, 2)}")

Both work inside `while` loops too — in fact `while True:` plus an `if ... break` is a common way to write "loop until this specific thing happens". Use these sparingly: one well-placed `break` makes code clearer, five make it a maze.

#### ✏️ Try It Yourself — Exercise 4
**Difficulty:** ★★★★☆  **Skills:** `continue`, `break`, accumulator

A survey recorded ages with some gaps: `ages = [34, None, 28, 45, None, 71, 19, 63]`.
(a) Using `continue` to skip the `None` entries, count the valid ages and compute their **mean**, rounded to 2 decimals.
(b) In a *second* loop, use `break` to find the **first** age above 60 and print it — then confirm your loop really stopped early by printing which ages it checked along the way.

In [22]:
# Your code here
ages = [34, None, 28, 45, None, 71, 19, 63]
count=0
total=0
for i in ages:
  if i is None:
    continue
  count=count+1
  total=total+i
print(count,total/len(ages))
for i in ages:
  if i is None:
    continue
  elif i>60:
    print(i)
    break

6 32.5
71


<details>
<summary>▶ Show solution</summary>

```python
ages = [34, None, 28, 45, None, 71, 19, 63]

# (a) skip the gaps, accumulate the rest
total = 0
count = 0
for age in ages:
    if age is None:
        continue
    total += age
    count += 1
print(f"{count} valid ages, mean {round(total / count, 2)}")

# (b) search with an early exit
for age in ages:
    if age is None:
        continue
    print("checking", age)
    if age > 60:
        print("first age above 60:", age)
        break        # 19 and 63 are never checked — the search ended at 71
```

</details>

### 2.8 Loops meet data — and when to vectorize instead
Now the payoff: combining today's tools with the pandas skills you already have. Load our two classical datasets back in.

In [ ]:
iris = pd.read_csv(IRIS_URL)
titanic = pd.read_csv(TITANIC_URL)
print("iris:", iris.shape, "| titanic:", titanic.shape)

**Good loop #1 — report lines per group.** In Session 04 you turned a `groupby` result into a dictionary with `.to_dict()`. Loop over its `.items()` and you can print a *sentence* per group — something no one-liner does gracefully.

In [23]:
fare_by_class = titanic.groupby("pclass")["fare"].mean().round(2).to_dict()
print(fare_by_class)

for pclass, fare in fare_by_class.items():
    print(f"class {pclass}: average fare ${fare}")

NameError: name 'titanic' is not defined

**Good loop #2 — branch on each group.** You do not even need the `.to_dict()` step: the `.mean()` on a grouped column hands back a **Series**, and a Series has `.items()` just like a dictionary — it yields `(group, value)` pairs directly. Add an `if` and each group gets a verdict.

In [ ]:
mean_petal = iris["petal_length"].mean()

for species, m in iris.groupby("species")["petal_length"].mean().items():
    if m > mean_petal:
        print(f"{species}: {round(m, 2)} cm — longer than the garden average")
    else:
        print(f"{species}: {round(m, 2)} cm — on the shorter side")

**Good loop #3 — walk the columns.** Column names are just an iterable of strings, so auditing a whole table takes four lines. This is the loop you will reuse on every new dataset you ever meet.

In [ ]:
# which columns have missing values, and how many?
for col in titanic.columns:
    n_missing = titanic[col].isna().sum()
    if n_missing > 0:
        print(f"{col}: {n_missing} missing")

#### ✏️ Try It Yourself — Exercise 5
**Difficulty:** ★★★★★  **Skills:** `groupby().to_dict()`, dict `.items()`, loop + `if`, f-strings, column audit

Write a tiny automated report on `titanic`:
(a) Build `survival_by_class = titanic.groupby("pclass")["survived"].mean().round(3).to_dict()`, and store the overall survival rate in `overall` (also the mean of a 0/1 column!).
(b) Loop over `survival_by_class.items()` and print one line per class, e.g. `class 1: 63.0% survived — above the ship average`, using an `if`/`else` against `overall`. *(Tip: `round(rate * 100, 1)` turns a proportion into a percent.)*
(c) Finish with a column audit: loop over `titanic.columns` and print `<column>: <n> missing` **only** for columns that have missing values.

In [ ]:
# Your code here
survival_by_class = titanic.groupby("pclass")["survived"].mean().round(3).to_dict()

<details>
<summary>▶ Show solution</summary>

```python
# (a) the summary, as a plain dictionary
survival_by_class = titanic.groupby("pclass")["survived"].mean().round(3).to_dict()
overall = titanic["survived"].mean()
print(survival_by_class, "| overall:", round(overall, 3))

# (b) one report line per class
for pclass, rate in survival_by_class.items():
    if rate > overall:
        print(f"class {pclass}: {round(rate * 100, 1)}% survived — above the ship average")
    else:
        print(f"class {pclass}: {round(rate * 100, 1)}% survived — below the ship average")

# (c) the audit loop from class, verbatim — it earns its keep on every dataset
for col in titanic.columns:
    n_missing = titanic[col].isna().sum()
    if n_missing > 0:
        print(f"{col}: {n_missing} missing")
```

</details>

### 3.1 Functions you already use — the built-ins
Look at what you already know how to call: `print`, `len`, `type`, `round`, `int`, `list`, `range`, `enumerate`, `zip`. Each one is a **function**: you hand it something in parentheses (the **arguments**), it does its job, and it hands something back (the **return value**). `len("iris")` takes a string and returns `4`. That is the whole idea, and everything today is a variation on it.

Python ships with a few dozen of these **built-in functions**, and we deliberately kept some of them out of sight until now — because last week you built them yourself. Here they are.

In [ ]:
scores = [88, 92, 79, 95, 84]

print("sum   :", sum(scores))          # last week's accumulator loop, in one word
print("min   :", min(scores), "| max:", max(scores))   # the best-so-far tracker, likewise
print("sorted:", sorted(scores))       # a NEW sorted list -- the original is untouched
print("abs   :", abs(-3.5))            # distance from zero
print("mean  :", round(sum(scores) / len(scores), 2))

Two things worth noticing. `sorted(scores)` **returns a new list** and leaves `scores` alone, unlike the `.sort()` method from Session 01, which reorders the list in place and returns nothing. And the mean is *still* not a built-in — you compose it from `sum` and `len`, which is a small preview of why you will want functions of your own.

Every function comes with documentation. Ask for it with `help(...)`, and read the first line: it tells you what goes in and what comes out.

In [ ]:
help(sorted)

Functions also live in **modules** — libraries you `import`, the way you have imported `json`, `os`, and `pandas`. The `math` module is the classic example: same idea, just reached through a dot.

In [ ]:
import math

print(math.sqrt(144))        # 12.0
print(math.floor(3.7))       # 3  -- round DOWN, like // from Session 02
print(math.pi)               # not a function: a constant the module carries around

# NumPy's versions do the same job on a whole array at once (Session 02's vectorization)
print(np.sqrt(np.array([1, 4, 9])))

### 3.2 `def` — your first function
Session 05's accumulator loop is back, this time wrapped in a function. Four parts: the keyword **`def`**, a **name** and **parameters** in parentheses followed by a colon, an indented **body**, and a **`return`** that hands the answer back.

In [24]:
def mean_of(values):
    """Return the arithmetic mean of a list of numbers."""
    total = 0
    for v in values:
        total += v
    return total / len(values)

print(mean_of(scores))          # the SAME code, now reusable...
print(mean_of([1, 2, 3, 4]))    # ...on any list you like

87.6
2.5


The line right under `def` — the string in triple quotes — is the **docstring**: a one-sentence description of what the function does. Docstrings are optional, and you should write one every time: `help()` prints it, and when you come back to the function months later it is the quickest way to recall what past-you was thinking.

Two moments matter, and beginners often blur them:
- **Defining** (`def ...`) teaches Python the recipe. Nothing runs yet — the body is stored, not executed.
- **Calling** (`mean_of(scores)`) actually runs the body with real arguments plugged into the parameters.

In [ ]:
print(mean_of)          # the recipe itself: a function object, not a number
print(mean_of.__doc__)  # its docstring, tucked inside
help(mean_of)           # what a user of your function sees

One naming note. **Parameters** are the placeholder names in the `def` line (`values`); **arguments** are the actual things you pass in the call (`scores`). People use the words loosely, and that is fine — but the distinction explains why `values` does not exist outside the function, which is where §2.5 is heading.

#### ✏️ Try It Yourself — Exercise 1
**Difficulty:** ★★☆☆☆  **Skills:** `def`, `return`, docstring

Write a function `pct_change(old, new)` that returns the **percent change** from `old` to `new`, rounded to 1 decimal — the formula is `(new - old) / old * 100`. Give it a docstring. Then check it: `pct_change(100, 120)` should be `20.0`, and `pct_change(80, 60)` should be `-25.0`. Finally, print its docstring with `.__doc__`.

In [ ]:
# Your code here

<details>
<summary>▶ Show solution</summary>

```python
def pct_change(old, new):
    """Percent change from old to new, rounded to 1 decimal."""
    return round((new - old) / old * 100, 1)

print(pct_change(100, 120))   # 20.0
print(pct_change(80, 60))     # -25.0
print(pct_change.__doc__)
```

</details>

---
## 4. Conclusions & Key Takeaways
- `if` / `elif` / `else` checks conditions from top to bottom and runs only the first branch that is true. Indentation shows what belongs to each branch.
- The conditional expression `x if cond else y` picks one of two values in one line. Both halves are required.
- A `for` loop walks through any iterable: a list, a string, a `range`, or the `.items()` of a dictionary or Series. A `while` loop repeats until its condition turns `False`, so something inside the loop has to change it.
- For an accumulator, you create the variable before the loop, update it inside, and use it after. Running totals, counters, and best-so-far trackers all work this way.
- `enumerate` gives you each position along with its value, and `zip` walks two lists side by side.
- `break` ends the whole loop, so a search can stop as soon as it finds its answer. `continue` skips just the current pass and is handy for stepping over bad entries. Both almost always sit inside an `if`.
- Loop over things like groups, columns, and report lines. For arithmetic on rows of numbers, stick with vectorized pandas and NumPy.
- A function takes arguments, does its job, and hands back a return value. `sum()`, `min()`, and `max()` do in one call what today's accumulator and best-so-far loops did by hand. `sorted()` returns a new list, while the `.sort()` method reorders the original in place.
- `help()` shows a function's documentation. Modules such as `math` hold more functions, which you reach with a dot: `math.sqrt(144)`.
- A function you write with `def` has a name, parameters, an indented body, and a `return`. Defining it only stores the recipe; the body runs when you call it, with your arguments filling in the parameters.
- Write a docstring for every function. `help()` prints it, and months later it reminds you what the function does.

### 🧾 Quick Reference
| Task | Code |
|---|---|
| branch | `if cond:` / `elif cond:` / `else:` |
| one-line pick | `"yes" if cond else "no"` |
| loop a list | `for x in xs:` |
| loop n times | `for i in range(n):` |
| positions + values | `for i, x in enumerate(xs, start=1):` |
| two lists in step | `for a, b in zip(xs, ys):` |
| loop a dict | `for k, v in d.items():` |
| repeat until | `while cond:` |
| running total | `total += x` |
| stop the loop / skip one pass | `break` / `continue` |
| loop a group summary | `for g, v in df.groupby("g")["x"].mean().items():` |
| audit columns | `for col in df.columns:` |
| handy built-ins | `sum(xs)`, `min(xs)`, `max(xs)`, `sorted(xs)`, `abs(x)` |
| read the docs | `help(sorted)` |
| function from a module | `import math`, then `math.sqrt(144)` |
| define a function | `def f(x):` … `return result` |
| docstring | first line inside `def`: `"""What it does."""` |
| call a function | `f(3)` |

### ⏭️ Coming Up Next  

I will post more details about the announcements below on Canvas：

1. This session's assignment is due on Nov. 2. You can find it in the `assignment/` folder or on the course GitHub, which I will update today.
2. The midterm is on Oct. 27, in person and on paper. Please arrive at least 5 minutes early. You may bring a one-sided cheat sheet and a calculator.
3. There is no live session on Tuesday, Nov. 3. Instead, I will post a recording that introduces Git, GitHub, the terminal, and the shell. *To make your life easier, the assignment for that session is optional.*
4. For the final project proposal, please email me to schedule a 30-minute meeting before Nov. 10 to discuss your idea. You do not need to submit a written proposal. I will give you feedback and suggestions during the meeting. No slides needed, just a conversation.

**More on functions.** Today you wrote your first function. Next session adds default and keyword arguments, functions that return more than one value, and scope, which explains why `values` does not exist outside `mean_of`. You will also meet the tiny anonymous `lambda`. Read McKinney Ch 3.

### 📌 This Week's Assignment
We have an assignment for this session, but the due is Nov.2nd. See the `assignment/` folder or the course GitHub (I will update it today).

> 💡 **Tip:** When a loop confuses you, put a `print()` inside it that shows the loop variable on each pass. Watching the values change is the fastest way to see what the loop is doing, and professional programmers debug loops the same way. Once a loop works and you catch yourself copying it into another cell, wrap it in a `def`, the way `mean_of` wraps the accumulator.